#ViT

In [43]:
from itertools import chain
from collections import defaultdict
from torch.utils.data import Subset
from torchvision import datasets

In [44]:
def subset_sampler(dataset, classes, max_len):
  target_idx = defaultdict(list)
  for idx, label in enumerate(dataset.targets):
    target_idx[int(label)].append(idx)

  indices = list(
      chain.from_iterable(
          [target_idx[idx][:max_len] for idx in range(len(classes))]
      )
  )
  return Subset(dataset, indices)

train_dataset = datasets.FashionMNIST(root='../datasets', download= True, train = True)
test_dataset = datasets.FashionMNIST(root='../datasets', download = True, train = False)

classes = train_dataset.classes
class_to_idx = train_dataset.class_to_idx

print(classes)
print(class_to_idx)

subset_train_dataset = subset_sampler(
    dataset = train_dataset, classes = train_dataset.classes, max_len = 100
)
subset_test_dataset = subset_sampler(
    dataset = test_dataset, classes = test_dataset.classes, max_len = 10
)

print(f"Training Data Size : {len(subset_train_dataset)}")
print(f"Testing Data Size : {len(subset_test_dataset)}")
print(train_dataset[0])

['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
{'T-shirt/top': 0, 'Trouser': 1, 'Pullover': 2, 'Dress': 3, 'Coat': 4, 'Sandal': 5, 'Shirt': 6, 'Sneaker': 7, 'Bag': 8, 'Ankle boot': 9}
Training Data Size : 1000
Testing Data Size : 100
(<PIL.Image.Image image mode=L size=28x28 at 0x7E2339BF91D0>, 9)


In [45]:
import torch
from torchvision import transforms
from transformers import AutoImageProcessor

image_processor = AutoImageProcessor.from_pretrained(
    pretrained_model_name_or_path = 'google/vit-base-patch16-224-in21k'
)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize(
        size = (image_processor.size['height'], image_processor.size['width'])
    ),
    transforms.Lambda(lambda x: torch.cat([x, x, x], 0)),
    transforms.Normalize(mean = image_processor.image_mean,
                        std = image_processor.image_std)
])

print(f"size : {image_processor.size}")
print(f"mean : {image_processor.image_mean}")
print(f"std : {image_processor.image_std}")

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


size : {'height': 224, 'width': 224}
mean : [0.5, 0.5, 0.5]
std : [0.5, 0.5, 0.5]


In [46]:
from torch.utils.data import DataLoader

# ViT 모델에 맞는 입력 구조 변환 수행
def collator(data, transform):
  images, labels = zip(*data)
  pixel_values = torch.stack([transform(image) for image in images])
  labels = torch.tensor([label for label in labels])
  return {'pixel_values': pixel_values, 'labels': labels} # dictionary 구조로 값 반환

train_dataloader = DataLoader(
    subset_train_dataset, batch_size = 32, shuffle = True,
    collate_fn = lambda x: collator(x, transform), drop_last = True
)
valid_dataloader = DataLoader(
    subset_test_dataset, batch_size = 4, shuffle = True,
    collate_fn = lambda x: collator(x, transform), drop_last = True
)

batch = next(iter(train_dataloader))
for key, value in batch.items():
  print(f'{key}: {value.shape}')

pixel_values: torch.Size([32, 3, 224, 224])
labels: torch.Size([32])


In [47]:
from transformers import ViTForImageClassification

model = ViTForImageClassification.from_pretrained(
    pretrained_model_name_or_path = 'google/vit-base-patch16-224-in21k',
    num_labels = len(classes),
    id2label = {idx: label for label, idx in class_to_idx.items()},
    label2id = class_to_idx,
    ignore_mismatched_sizes = True
)

print(model.classifier)

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Linear(in_features=768, out_features=10, bias=True)


In [48]:
print(model.vit.embeddings)

batch = next(iter(train_dataloader))
print ("image shape :", batch["pixel_values"]. shape)
print("patch enbeddings shape :",
model.vit. embeddings.patch_embeddings(batch["pixel_values"]).shape)
print('[CLS] + patch.embeddings.shape:',
model.vit.embeddings(batch['pixel_values']).shape)

ViTEmbeddings(
  (patch_embeddings): ViTPatchEmbeddings(
    (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  )
  (dropout): Dropout(p=0.0, inplace=False)
)
image shape : torch.Size([32, 3, 224, 224])
patch enbeddings shape : torch.Size([32, 196, 768])
[CLS] + patch.embeddings.shape: torch.Size([32, 197, 768])


In [49]:
from transformers import TrainingArguments
args = TrainingArguments(
    output_dir = '../models/ViT-FashionMNIST',
    save_strategy = 'epoch',
    eval_strategy = 'epoch',
    learning_rate = 1e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 3,
    weight_decay = 0.01,
    load_best_model_at_end = True,
    metric_for_best_model = 'f1',
    logging_dir = 'logs',
    logging_steps = 125,
    remove_unused_columns = False,
    seed = 7
)

In [50]:
import evaluate
import numpy as np

def compute_metrics(eval_pred):
  metric = evaluate.load('f1')
  predictions, labels = eval_pred
  predictions = np.argmax(predictions, axis = 1)
  macro_f1 = metric.compute(
      predictions = predictions, references = labels, average = 'macro'
  )
  return macro_f1

In [51]:
import torch
import evaluate
import numpy as np
from itertools import chain
from collections import defaultdict
from torch.utils.data import Subset
from torchvision import datasets
from torchvision import transforms
from transformers import AutoImageProcessor
from transformers import ViTForImageClassification
from transformers import TrainingArguments, Trainer

In [ ]:
def model_init(classes, class_to_idx):
  model = ViTForImageClassification.from_pretrained(
    pretrained_model_name_or_path="google/vit-base-patch16-224-in21k",
    num_labels= len(classes),
    id2label={idx: label for label, idx in class_to_idx.items()},
    label2id=class_to_idx,
  )
  return model

trainer = Trainer(
    model_init = lambda x: model_init(classes, class_to_idx),
    args = args,
    train_dataset = subset_train_dataset,
    eval_dataset = subset_test_dataset,
    data_collator = lambda x: collator(x, transform),
    compute_metrics = compute_metrics,
    tokenizer = image_processor
)
trainer.train()

<ipython-input-52-7b395d8ede0f>:10: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

outputs = trainer.predict(subset_test_dataset)
print(outputs)

y_true = outputs.label_ids
y_pred = outputs.predictions.argmax(1)

labels= list(classes)
matrix = confusion_matrix(y_true, y_pred)
display = ConfusionMatrixDisplay(confusion_matrix = matrix, display_labels = labels)
_, ax = plt.subplots(figsize = (10, 10))
display.plot(xticks_rotation = 45, ax = ax)
plt.show()

#Swin Transformer

In [14]:
import torch

window_size = 2
coords_h = torch.arange(window_size)
coords_w = torch.arange(window_size)
coords = torch.stack(torch.meshgrid([coords_h, coords_w], indexing = 'ij'))
coords_flatten = torch.flatten(coords, 1)
relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
print(relative_coords)
print(relative_coords.shape)

tensor([[[ 0,  0, -1, -1],
         [ 0,  0, -1, -1],
         [ 1,  1,  0,  0],
         [ 1,  1,  0,  0]],

        [[ 0, -1,  0, -1],
         [ 1,  0,  1,  0],
         [ 0, -1,  0, -1],
         [ 1,  0,  1,  0]]])
torch.Size([2, 4, 4])


In [15]:
x_coords = relative_coords[0, : , : ]
y_coords = relative_coords[1, : , : ]

x_coords += window_size - 1
y_coords += window_size - 1
x_coords *= 2* window_size - 1
print(f"X축에 대한 행렬 :\n{x_coords}\n")
print(f"Y축에 대한 행렬:\n{y_coords}\n")

relative_position_index =x_coords + y_coords
print(f'X, Y 축에 대한 위치 행렬:\n{relative_position_index}')

X축에 대한 행렬 :
tensor([[3, 3, 0, 0],
        [3, 3, 0, 0],
        [6, 6, 3, 3],
        [6, 6, 3, 3]])

Y축에 대한 행렬:
tensor([[1, 0, 1, 0],
        [2, 1, 2, 1],
        [1, 0, 1, 0],
        [2, 1, 2, 1]])

X, Y 축에 대한 위치 행렬:
tensor([[4, 3, 1, 0],
        [5, 4, 2, 1],
        [7, 6, 4, 3],
        [8, 7, 5, 4]])


In [16]:
num_heads = 1
relative_position_bias_table = torch.Tensor(
torch.zeros ((2 * window_size - 1) * (2 * window_size - 1), num_heads))

relative_position_bias = relative_position_bias_table[relative_position_index.view(-1)]
relative_position_bias = relative_position_bias.view(
window_size * window_size, window_size * window_size, -1
)
print(relative_position_bias.shape)

torch.Size([4, 4, 1])


In [18]:
from transformers import SwinForImageClassification

In [20]:
from transformers import SwinForImageClassification

model = SwinForImageClassification.from_pretrained(
    pretrained_model_name_or_path = 'microsoft/swin-tiny-patch4-window7-224',
    num_labels = len(train_dataset.classes),
    id2label = {idx: label for label, idx in train_dataset.class_to_idx.items()},
    label2id = train_dataset.class_to_idx,
    ignore_mismatched_sizes = True
)

for main_name, main_module in model.named_children():
  print(main_name)
  for sub_name, sub_module in main_module.named_children():
    print("L", sub_name)
    for ssub_name, ssub_module in sub_module.named_children():
      print("| L", ssub_name)
      for sssub_name, sssub_module in ssub_module.named_children():
        if ssub_name == 'projection':
          print("| | L", sssub_name, sssub_module)
        else:
          print("| | L", sssub_name)

config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/113M [00:00<?, ?B/s]

Some weights of SwinForImageClassification were not initialized from the model checkpoint at microsoft/swin-tiny-patch4-window7-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


swin
L embeddings
| L patch_embeddings
| | L projection
| L norm
| L dropout
L encoder
| L layers
| | L 0
| | L 1
| | L 2
| | L 3
L layernorm
L pooler
classifier


In [21]:
batch = next(iter(train_dataloader))
print("이미지 차원:", batch['pixel_values'].shape)

patch_emb_output, shape = model.swin.embeddings.patch_embeddings(batch['pixel_values'])
print("모듈:", model.swin.embeddings.patch_embeddings)
print("패치 임베딩 차원:", patch_emb_output.shape)

이미지 차원: torch.Size([32, 3, 224, 224])
모듈: SwinPatchEmbeddings(
  (projection): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
)
패치 임베딩 차원: torch.Size([32, 3136, 96])


In [23]:
for main_name, main_module in model.swin.encoder.layers[0].named_children():
  print(main_name)
  for sub_name, sub_module in main_module.named_children():
    print('L', sub_name)
    for ssub_name, ssub_module in sub_module.named_children():
      print("| L", ssub_name)

blocks
L 0
| L layernorm_before
| L attention
| L drop_path
| L layernorm_after
| L intermediate
| L output
L 1
| L layernorm_before
| L attention
| L drop_path
| L layernorm_after
| L intermediate
| L output
downsample
L reduction
L norm


In [24]:
print(model.swin.encoder.layers[0].blocks[0])

SwinLayer(
  (layernorm_before): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  (attention): SwinAttention(
    (self): SwinSelfAttention(
      (query): Linear(in_features=96, out_features=96, bias=True)
      (key): Linear(in_features=96, out_features=96, bias=True)
      (value): Linear(in_features=96, out_features=96, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (output): SwinSelfOutput(
      (dense): Linear(in_features=96, out_features=96, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
  )
  (drop_path): Identity()
  (layernorm_after): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  (intermediate): SwinIntermediate(
    (dense): Linear(in_features=96, out_features=384, bias=True)
    (intermediate_act_fn): GELUActivation()
  )
  (output): SwinOutput(
    (dense): Linear(in_features=384, out_features=96, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
)


In [25]:
print("패치 임베딩 차원 :", patch_emb_output.shape)

W_MSA = model.swin.encoder.layers[0].blocks[0]
SW_MSA = model.swin.encoder.layers[0].blocks[1]

W_MSA_output = W_MSA(patch_emb_output, W_MSA.input_resolution)[0]
SW_MSA_output = SW_MSA(W_MSA_output, SW_MSA.input_resolution)[0]

print("W-NSA 결과 차원 :", W_MSA_output.shape)
print("SM-MSA 결과 차원 :", SW_MSA_output.shape)

패치 임베딩 차원 : torch.Size([32, 3136, 96])
W-NSA 결과 차원 : torch.Size([32, 3136, 96])
SM-MSA 결과 차원 : torch.Size([32, 3136, 96])


In [27]:
patch_merge = model.swin.encoder.layers[0].downsample
print("patch_merge 모듈:", patch_merge)

output = patch_merge(SW_MSA_output, patch_merge.input_resolution)
print("patch_merge 결과 차원:", output.shape)

patch_merge 모듈: SwinPatchMerging(
  (reduction): Linear(in_features=384, out_features=192, bias=False)
  (norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
)
patch_merge 결과 차원: torch.Size([32, 784, 192])


In [30]:
from transformers import SwinForImageClassification

def subset_sampler(dataset, classes, max_len):
  target_idx = defaultdict(list)
  for idx, label in enumerate(dataset.targets):
    target_idx[int(label)].append(idx)

  indices = list(
      chain.from_iterable(
          [target_idx[idx][:max_len] for idx in range(len(classes))]
      )
  )
  return Subset(dataset, indices)

def model_init(classes, class_to_idx):
  model = SwinForImageClassification.from_pretrained(
    pretrained_model_name_or_path="microsoft/swin-tiny-patch4-window7-224",
    num_labels= len(classes),
    id2label={idx: label for label, idx in class_to_idx.items()},
    label2id=class_to_idx,
    ignore_mismatched_sizes = True
  )
  return model

def collator(data, transform):
  images, labels = zip(*data)
  pixel_values = torch.stack([transform(image) for image in images])
  labels = torch.tensor([label for label in labels])
  return {'pixel_values': pixel_values, 'labels': labels} # dictionary 구조로 값 반환

def compute_metrics(eval_pred):
  metric = evaluate.load('f1')
  predictions, labels = eval_pred
  predictions = np.argmax(predictions, axis = 1)
  macro_f1 = metric.compute(
      predictions = predictions, references = labels, average = 'macro'
  )
  return macro_f1

train_dataset = datasets.FashionMNIST(root='../datasets', download= True, train = True)
test_dataset = datasets.FashionMNIST(root='../datasets', download = True, train = False)

classes = train_dataset.classes
class_to_idx = train_dataset.class_to_idx

print(classes)
print(class_to_idx)

subset_train_dataset = subset_sampler(
    dataset = train_dataset, classes = train_dataset.classes, max_len = 100
)
subset_test_dataset = subset_sampler(
    dataset = test_dataset, classes = test_dataset.classes, max_len = 10
)

image_processor = AutoImageProcessor.from_pretrained(
    pretrained_model_name_or_path = "microsoft/swin-tiny-patch4-window7-224"
)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize(
        size = (image_processor.size['height'], image_processor.size['width'])
    ),
    transforms.Lambda(lambda x: torch.cat([x, x, x], 0)),
    transforms.Normalize(mean = image_processor.image_mean,
                        std = image_processor.image_std)
])

args = TrainingArguments(
    output_dir = '../models/Swin-FashionMNIST',
    save_strategy = 'epoch',
    eval_strategy = 'epoch',
    learning_rate = 1e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 3,
    weight_decay = 0.001,
    load_best_model_at_end = True,
    metric_for_best_model = 'f1',
    logging_dir = 'logs',
    logging_steps = 125,
    remove_unused_columns = False,
    seed = 7
)

trainer = Trainer(
    model_init = lambda x: model_init(classes, class_to_idx),
    args = args,
    train_dataset = subset_train_dataset,
    eval_dataset = subset_test_dataset,
    data_collator = lambda x: collator(x, transform),
    compute_metrics = compute_metrics,
    tokenizer = image_processor
)

trainer.train()

['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
{'T-shirt/top': 0, 'Trouser': 1, 'Pullover': 2, 'Dress': 3, 'Coat': 4, 'Sandal': 5, 'Shirt': 6, 'Sneaker': 7, 'Bag': 8, 'Ankle boot': 9}


<ipython-input-30-1d39067c9350>:87: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Some weights of SwinForImageClassification were not initialized from the model checkpoint at microsoft/swin-tiny-patch4-window7-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of SwinForImageClassification were not initialized from the model checkpoint at microsoft/swin-tiny-patch4-window7-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and 

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

# CVT

In [36]:
def subset_sampler(dataset, classes, max_len):
  target_idx = defaultdict(list)
  for idx, label in enumerate(dataset.targets):
    target_idx[int(label)].append(idx)

  indices = list(
      chain.from_iterable(
          [target_idx[idx][:max_len] for idx in range(len(classes))]
      )
  )
  return Subset(dataset, indices)

train_dataset = datasets.FashionMNIST(root='../datasets', download= True, train = True)
test_dataset = datasets.FashionMNIST(root='../datasets', download = True, train = False)

# classes 메서드로 데이터셋에 포함된 클래스를 확인할 수 있음
# class_to_idx로 클래스 ID와 클래스가 매핑된 값 확인할 수 있음
classes = train_dataset.classes
class_to_idx = train_dataset.class_to_idx


# subset_sanpler로 섭샘플링 (dataset, classes(클래스 목록), max_len(클래스별 최대 샘플링 개수))
subset_train_dataset = subset_sampler(
    dataset = train_dataset, classes = train_dataset.classes, max_len = 6000
)
subset_test_dataset = subset_sampler(
    dataset = test_dataset, classes = test_dataset.classes, max_len = 1000
)

import torch
from torchvision import transforms
from transformers import AutoImageProcessor

image_processor = AutoImageProcessor.from_pretrained(
    pretrained_model_name_or_path = 'microsoft/cvt-21'
)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize(
        size = (
        image_processor.size['shortest_edge'],
        image_processor.size['shortest_edge']
        )
    ),
    transforms.Lambda(lambda x: torch.cat([x, x, x], 0)),
    transforms.Normalize(mean = image_processor.image_mean,
                        std = image_processor.image_std)
])

from torch.utils.data import DataLoader
def collator(data, transform):
  images, labels = zip(*data)
  pixel_values = torch.stack([transform(image) for image in images])
  labels = torch.tensor([label for label in labels])
  return {'pixel_values': pixel_values, 'labels': labels}
  # dictionary 구조로 값 반환

train_dataloader = DataLoader(
    subset_train_dataset, batch_size = 32, shuffle = True,
    collate_fn = lambda x: collator(x, transform), drop_last = True
)
valid_dataloader = DataLoader(
    subset_test_dataset, batch_size = 4, shuffle = True,
    collate_fn = lambda x: collator(x, transform), drop_last = True
)

batch = next(iter(train_dataloader))
for key, value in batch.items():
  print(f'{key}: {value.shape}')

pixel_values: torch.Size([32, 3, 224, 224])
labels: torch.Size([32])


In [37]:
from transformers import CvtForImageClassification

model = CvtForImageClassification.from_pretrained(
    pretrained_model_name_or_path = 'microsoft/cvt-21',
    num_labels = len(train_dataset.classes),
    id2label = {idx: label for label, idx in train_dataset.class_to_idx.items()},
    label2id = train_dataset.class_to_idx,
    ignore_mismatched_sizes = True
)

for main_name, main_module in model.named_children():
  print(main_name)
  for sub_name, sub_module in main_module.named_children():
    print('L', sub_name)
    for ssub_name, ssub_module in sub_module.named_children():
      print("   L", ssub_name)
      for sssub_name, sssub_module in ssub_module.named_children():
        print("     L", sssub_name)

config.json:   0%|          | 0.00/70.3k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/127M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/127M [00:00<?, ?B/s]

Some weights of CvtForImageClassification were not initialized from the model checkpoint at microsoft/cvt-21 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 384]) in the checkpoint and torch.Size([10, 384]) in the model instantiated
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([10]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


cvt
L encoder
   L stages
     L 0
     L 1
     L 2
layernorm
classifier


In [38]:
stages = model.cvt.encoder.stages
print(stages[0])

CvtStage(
  (embedding): CvtEmbeddings(
    (convolution_embeddings): CvtConvEmbeddings(
      (projection): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(2, 2))
      (normalization): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (layers): Sequential(
    (0): CvtLayer(
      (attention): CvtAttention(
        (attention): CvtSelfAttention(
          (convolution_projection_query): CvtSelfAttentionProjection(
            (convolution_projection): CvtSelfAttentionConvProjection(
              (convolution): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
              (normalization): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            )
            (linear_projection): CvtSelfAttentionLinearProjection()
          )
          (convolution_projection_key): CvtSelfAttentionProjection(
            (convolution_projection): CvtSelfAtte

In [41]:
batch = next(iter(train_dataloader))
print(" 이미지 차원 :", batch["pixel_values"].shape)

patch_emb_output = stages[0].embedding(batch["pixel_values"])
print(" 패치 임베딩 차원 :", patch_emb_output.shape)

batch_size, num_channels, height, width = patch_emb_output.shape
hidden_state = patch_emb_output.view(batch_size, num_channels, height * width).permute(0, 2, 1)
print('셀프 어텐션 입력 차원: ', hidden_state.shape)

attention_output = stages[0].layers[0].attention.attention(hidden_state, height, width)
print(" 셀프 어텐션 출력 차원 : " , attention_output.shape)

 이미지 차원 : torch.Size([32, 3, 224, 224])
 패치 임베딩 차원 : torch.Size([32, 64, 56, 56])
셀프 어텐션 입력 차원:  torch.Size([32, 3136, 64])
 셀프 어텐션 출력 차원 :  torch.Size([32, 3136, 64])


In [42]:
from transformers import CvtForImageClassification

def subset_sampler(dataset, classes, max_len):
  target_idx = defaultdict(list)
  for idx, label in enumerate(dataset.targets):
    target_idx[int(label)].append(idx)

  indices = list(
      chain.from_iterable(
          [target_idx[idx][:max_len] for idx in range(len(classes))]
      )
  )
  return Subset(dataset, indices)

def model_init(classes, class_to_idx):
  model = CvtForImageClassification.from_pretrained(
    pretrained_model_name_or_path="microsoft/cvt-21",
    num_labels= len(classes),
    id2label={idx: label for label, idx in class_to_idx.items()},
    label2id=class_to_idx,
    ignore_mismatched_sizes = True
  )
  return model

def collator(data, transform):
  images, labels = zip(*data)
  pixel_values = torch.stack([transform(image) for image in images])
  labels = torch.tensor([label for label in labels])
  return {'pixel_values': pixel_values, 'labels': labels} # dictionary 구조로 값 반환

def compute_metrics(eval_pred):
  metric = evaluate.load('f1')
  predictions, labels = eval_pred
  predictions = np.argmax(predictions, axis = 1)
  macro_f1 = metric.compute(
      predictions = predictions, references = labels, average = 'macro'
  )
  return macro_f1

train_dataset = datasets.FashionMNIST(root='../datasets', download= True, train = True)
test_dataset = datasets.FashionMNIST(root='../datasets', download = True, train = False)

classes = train_dataset.classes
class_to_idx = train_dataset.class_to_idx

subset_train_dataset = subset_sampler(
    dataset = train_dataset, classes = train_dataset.classes, max_len = 100
)
subset_test_dataset = subset_sampler(
    dataset = test_dataset, classes = test_dataset.classes, max_len = 10
)

image_processor = AutoImageProcessor.from_pretrained(
    pretrained_model_name_or_path = "microsoft/cvt-21"
)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize(
        size = (image_processor.size['shortest_edge'], image_processor.size['shortest_edge'])
    ),
    transforms.Lambda(lambda x: torch.cat([x, x, x], 0)),
    transforms.Normalize(mean = image_processor.image_mean,
                        std = image_processor.image_std)
])

args = TrainingArguments(
    output_dir = '../models/CvT-FashionMNIST',
    save_strategy = 'epoch',
    eval_strategy = 'epoch',
    learning_rate = 1e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 3,
    weight_decay = 0.001,
    load_best_model_at_end = True,
    metric_for_best_model = 'f1',
    logging_dir = 'logs',
    logging_steps = 125,
    remove_unused_columns = False,
    seed = 7
)

trainer = Trainer(
    model_init = lambda x: model_init(classes, class_to_idx),
    args = args,
    train_dataset = subset_train_dataset,
    eval_dataset = subset_test_dataset,
    data_collator = lambda x: collator(x, transform),
    compute_metrics = compute_metrics,
    tokenizer = image_processor
)

trainer.train()

<ipython-input-42-cc30f1262ecd>:84: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Some weights of CvtForImageClassification were not initialized from the model checkpoint at microsoft/cvt-21 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 384]) in the checkpoint and torch.Size([10, 384]) in the model instantiated
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([10]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of CvtForImageClassification were not initialized from the model checkpoint at microsoft/cvt-21 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 384]) in the checkpoint and torch.Size([10, 384]) in the model inst

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

outputs = trainer.predict(subset_test_dataset)
print(outputs)

y_true = outputs.label_ids
y_pred = outputs.predictions.argmax(1)

labels= list(classes)
matrix = confusion_matrix(y_true, y_pred)
display = ConfusionMatrixDisplay(confusion_matrix = matrix, display_labels = labels)
_, ax = plt.subplots(figsize = (10, 10))
display.plot(xticks_rotation = 45, ax = ax)
plt.show()